In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Konstruowanie Circuit

<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie został opracowany z użyciem poniższych wymagań.
Zalecamy korzystanie z tych wersji lub nowszych.

```
qiskit[all]~=2.3.0
```
</details>
Ta strona dokładniej omawia klasę [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) w SDK Qiskit, w tym kilka bardziej zaawansowanych metod, których możesz użyć do tworzenia kwantowych Circuit.
## Czym jest kwantowy Circuit?
Prosty kwantowy Circuit to zbiór Qubitów i lista instrukcji działających na tych Qubitach. Aby to zobrazować, poniższa komórka tworzy nowy Circuit z dwoma nowymi Qubitami, a następnie wyświetla atrybut [`qubits`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qubits) tego Circuit, który jest listą obiektów [`Qubit`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Qubit) w kolejności od najmniej znaczącego bitu $q_0$ do najbardziej znaczącego bitu $q_n$.

In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.qubits

[<Qubit register=(2, "q"), index=0>, <Qubit register=(2, "q"), index=1>]

Multiple `QuantumRegister` and `ClassicalRegister` objects can be combined to create a circuit. Every [`QuantumRegister`](/docs/api/qiskit/circuit#qiskit.circuit.QuantumRegister) and [`ClassicalRegister`](/docs/api/qiskit/circuit#qiskit.circuit.ClassicalRegister) can also be named.

In [2]:
from qiskit.circuit import QuantumRegister, ClassicalRegister

qr1 = QuantumRegister(2, "qreg1")  # Create a QuantumRegister with 2 qubits
qr2 = QuantumRegister(1, "qreg2")  # Create a QuantumRegister with 1 qubit
cr1 = ClassicalRegister(3, "creg1")  # Create a ClassicalRegister with 3 cbits

combined_circ = QuantumCircuit(
    qr1, qr2, cr1
)  # Create a quantum circuit with 2 QuantumRegisters and 1 ClassicalRegister
combined_circ.qubits

[<Qubit register=(2, "qreg1"), index=0>,
 <Qubit register=(2, "qreg1"), index=1>,
 <Qubit register=(1, "qreg2"), index=0>]

Wiele obiektów `QuantumRegister` i `ClassicalRegister` można łączyć, aby tworzyć Circuit. Każdy obiekt [`QuantumRegister`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.QuantumRegister) i [`ClassicalRegister`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.ClassicalRegister) może również mieć nazwę.

In [3]:
desired_qubit = qr2[0]  # Qubit 0 of register 'qreg2'

print("Index:", combined_circ.find_bit(desired_qubit).index)
print("Register:", combined_circ.find_bit(desired_qubit).registers)

Index: 2
Register: [(QuantumRegister(1, 'qreg2'), 0)]


Adding an instruction to the circuit appends the instruction to the circuit's [`data`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#data) attribute. The following cell output shows `data` is a list of [`CircuitInstruction`](/docs/api/qiskit/qiskit.circuit.CircuitInstruction) objects, each of which has an `operation` attribute, and a `qubits` attribute.

In [4]:
qc.x(0)  # Add X-gate to qubit 0
qc.data

[CircuitInstruction(operation=Instruction(name='x', num_qubits=1, num_clbits=0, params=[]), qubits=(<Qubit register=(2, "q"), index=0>,), clbits=())]

Indeks i rejestr Qubita możesz znaleźć, korzystając z metody [`find_bit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.find_bit) Circuit i jej atrybutów.

In [5]:
qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/43a57258-3e33-4071-8a48-2bf127c8a5be-0.svg" alt="Output of the previous code cell" />

Circuit instruction objects can contain "definition" circuits that describe the instruction in terms of more fundamental instructions. For example, the [X-gate](/docs/api/qiskit/qiskit.circuit.library.XGate) is defined as a specific case of the [U3-gate](/docs/api/qiskit/qiskit.circuit.library.U3Gate), a more general single-qubit gate.

In [6]:
# Draw definition circuit of 0th instruction in `qc`
qc.data[0].operation.definition.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/653e2427-e301-4d2f-84de-1959185ace8e-0.svg" alt="Output of the previous code cell" />

Dodanie instrukcji do Circuit dołącza ją do atrybutu [`data`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#data) tego Circuit. Poniższe dane wyjściowe komórki pokazują, że `data` jest listą obiektów [`CircuitInstruction`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.CircuitInstruction), z których każdy ma atrybut `operation` oraz atrybut `qubits`.

In [7]:
from qiskit.circuit.library import HGate

qc = QuantumCircuit(1)
qc.append(
    HGate(),  # New HGate instruction
    [0],  # Apply to qubit 0
)
qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/66813cae-9841-47ea-96b7-8fd7b82e9759-0.svg" alt="Output of the previous code cell" />

To combine two circuits, use the [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method. This accepts another [`QuantumCircuit`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit) and an optional list of qubit mappings.

<Admonition type="note">
    The [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method returns a new circuit and does **not** mutate either circuit it acts on. To mutate the circuit on which you're calling the [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method, use the argument `inplace=True`.
</Admonition>

In [8]:
qc_a = QuantumCircuit(4)
qc_a.x(0)

qc_b = QuantumCircuit(2, name="qc_b")
qc_b.y(0)
qc_b.z(1)

# compose qubits (0, 1) of qc_a to qubits (1, 3) of qc_b respectively
combined = qc_a.compose(qc_b, qubits=[1, 3])
combined.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/29152dfa-2275-4bc4-aadb-82185b9e0e86-0.svg" alt="Output of the previous code cell" />

Najprostszym sposobem wyświetlenia tych informacji jest metoda [`draw`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#draw), która zwraca wizualizację Circuit. Zobacz stronę [Wizualizacja Circuit](./visualize-circuits), aby poznać różne sposoby wyświetlania kwantowych Circuit.

In [9]:
inst = qc_b.to_instruction()
qc_a.append(inst, [1, 3])
qc_a.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/81b682dd-45cb-4492-809e-d9e8ebbf5600-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/43a57258-3e33-4071-8a48-2bf127c8a5be-0.svg)

Obiekty instrukcji Circuit mogą zawierać „definicyjne" Circuit opisujące instrukcję za pomocą bardziej podstawowych instrukcji. Na przykład [Gate X](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.XGate) jest zdefiniowany jako szczególny przypadek [Gate U3](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.U3Gate), będącego bardziej ogólnym Gate dla jednego Qubita.

In [10]:
gate = qc_b.to_gate().control()
qc_a.append(gate, [0, 1, 3])
qc_a.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/ed362e64-d6a4-4dfd-a5cf-5e6bdc7a81b5-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/653e2427-e301-4d2f-84de-1959185ace8e-0.svg)

Instrukcje i Circuit są podobne w tym sensie, że oba opisują operacje na bitach i Qubitach, ale służą różnym celom:

- Instrukcje są traktowane jako stałe, a ich metody zazwyczaj zwracają nowe instrukcje (bez modyfikowania oryginalnego obiektu).
- Circuit są zaprojektowane tak, aby budować je w wielu liniach kodu, a metody [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) często modyfikują istniejący obiekt.
### Czym jest głębokość Circuit?
[Głębokość (depth())](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.depth) kwantowego Circuit to miara liczby „warstw" Gate kwantowych wykonywanych równolegle, potrzebnych do ukończenia obliczeń zdefiniowanych przez ten Circuit. Ponieważ implementacja Gate kwantowych zajmuje czas, głębokość Circuit w przybliżeniu odpowiada ilości czasu potrzebnej komputerowi kwantowemu na wykonanie tego Circuit. Dlatego głębokość Circuit jest jedną z ważnych wielkości używanych do oceny, czy dany Circuit może być uruchomiony na urządzeniu.

Pozostała część tej strony ilustruje, jak manipulować kwantowymi Circuit.
## Budowanie Circuit
Metody takie jak [`QuantumCircuit.h`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#h) i [`QuantumCircuit.cx`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#cx) dodają konkretne instrukcje do Circuit. Aby bardziej ogólnie dodawać instrukcje do Circuit, użyj metody [`append`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#append). Przyjmuje ona instrukcję i listę Qubitów, do których ma być zastosowana. Zobacz [dokumentację API biblioteki Circuit](https://docs.quantum.ibm.com/api/qiskit/circuit_library), aby zapoznać się z listą obsługiwanych instrukcji.

In [11]:
qc_a.decompose().draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/3c0633db-929b-4428-a888-7a3d493bd6dd-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/66813cae-9841-47ea-96b7-8fd7b82e9759-0.svg)

Aby połączyć dwa Circuit, użyj metody [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose). Przyjmuje ona inny obiekt [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) oraz opcjonalną listę mapowań Qubitów.

> **Note:** Metoda [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose) zwraca nowy Circuit i **nie** modyfikuje żadnego z Circuit, na których działa. Aby zmodyfikować Circuit, na którym wywołujesz metodę [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose), użyj argumentu `inplace=True`.

In [12]:
qc1 = QuantumCircuit(2, 2)
qc1.measure(0, 1)
qc1.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/0cdb2273-0.svg" alt="Output of the previous code cell" />

In [13]:
qc2 = QuantumCircuit(2)
qc2.measure_all()
qc2.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/6f33698c-0.svg" alt="Output of the previous code cell" />

In [14]:
qc3 = QuantumCircuit(2)
qc3.x(1)
qc3.measure_active()
qc3.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/ca3f225f-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/ed362e64-d6a4-4dfd-a5cf-5e6bdc7a81b5-0.svg)

Aby zobaczyć, co się dzieje, możesz użyć metody [`decompose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#decompose), która rozbudowuje każdą instrukcję do jej definicji.

> **Note:** Metoda [`decompose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#decompose) zwraca nowy Circuit i **nie** modyfikuje Circuit, na którym działa.

In [15]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.circuit import Parameter

angle = Parameter("angle")  # undefined number

# Create and optimize circuit once
qc = QuantumCircuit(1)
qc.rx(angle, 0)
qc = generate_preset_pass_manager(
    optimization_level=3, basis_gates=["u", "cx"]
).run(qc)

qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/a580552c-d585-4047-99f0-32aafd06e4f3-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/3c0633db-929b-4428-a888-7a3d493bd6dd-0.svg)

<span id="measure-qubits"></span>
## Pomiar Qubitów
Pomiary służą do próbkowania stanów poszczególnych Qubitów i przesyłania wyników do rejestru klasycznego. Pamiętaj, że jeśli przesyłasz Circuit do prymitywu [Sampler](./primitives#sampler), pomiary są wymagane. Natomiast Circuit przesyłane do prymitywu [Estimator](./primitives#estimator) nie mogą zawierać pomiarów.

Qubity można mierzyć trzema metodami: [`measure`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.measure), [`measure_all`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) i [`measure_active`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_active). Aby dowiedzieć się, jak wizualizować wyniki pomiarów, zobacz stronę [Wizualizacja wyników](./visualize-results).

1. `QuantumCircuit.measure` : mierzy każdy Qubit podany w pierwszym argumencie na klasyczny bit wskazany jako drugi argument. Ta metoda pozwala na pełną kontrolę nad tym, gdzie jest przechowywany wynik pomiaru.

2. `QuantumCircuit.measure_all` : nie przyjmuje żadnego argumentu i może być używana dla kwantowych Circuit bez wcześniej zdefiniowanych bitów klasycznych. Tworzy klasyczne przewody i przechowuje wyniki pomiarów w kolejności. Na przykład pomiar Qubita $q_i$ jest przechowywany w cbicie $meas_i$). Dodaje również barierę przed pomiarem.

3. `QuantumCircuit.measure_active` : podobna do `measure_all`, ale mierzy tylko Qubity, które mają operacje.

In [16]:
circuits = []
for value in range(100):
    circuits.append(qc.assign_parameters({angle: value}))

circuits[0].draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/85af6231-921a-4130-99d3-f6998f761df8-0.svg" alt="Output of the previous code cell" />

You can find a list of a circuit's undefined parameters in its `parameters` attribute.

In [17]:
qc.parameters

ParameterView([Parameter(angle)])

### Change a parameter's name

By default, parameter names for a parameterized circuit are prefixed by `x`- for example, `x[0]`. You can change the names after they are defined, as shown in the following example.

In [18]:
from qiskit.circuit.library import z_feature_map
from qiskit.circuit import ParameterVector

# Define a parameterized circuit with default names
# For example, x[0]
circuit = z_feature_map(2)

# Set new parameter names
# They will now be prefixed by `hi` instead
# For example, hi[0]
training_params = ParameterVector("hi", 2)

# Assign parameter names to the quantum circuit
circuit = circuit.assign_parameters(parameters=training_params)

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/ca3f225f-0.svg)

## Parametryzowane Circuit

Wiele kwantowych algorytmów krótkoterminowych polega na wykonywaniu wielu wariantów kwantowego Circuit. Ponieważ konstruowanie i optymalizowanie dużych Circuit może być kosztowne obliczeniowo, Qiskit obsługuje Circuit **parametryzowane**. Takie Circuit mają niezdefiniowane parametry, których wartości nie trzeba podawać aż do tuż przed wykonaniem Circuit. Dzięki temu można przenieść konstruowanie i optymalizowanie Circuit poza główną pętlę programu. Poniższa komórka tworzy i wyświetla parametryzowany Circuit.